In [1]:
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 44.6 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [17]:
folder = '/content/drive/MyDrive/TransformesCod/03-restaurant/Chatbot_RAG/'
QA_file = folder + 'QA_file.csv'
QAfaiss_file = folder + 'QAfaiss_file.bin'

In [8]:
import pandas as pd
df = pd.read_csv('/content/drive/MyDrive/TransformesCod/03-restaurant/Chatbot_RAG/ indian_food.csv')
df.head(1)

,name,ARname,ingredients,ARingredients,diet,ARdiet,prep_time,cook_time,flavor_profile,course,ARflavor_profile,ARcourse,state,region
0,Balu shahi,بالو شاهي,"Maida flour, yogurt, oil, sugar",دقيق مائدة، زبادي، زيت، سكر,vegetarian,نباتي,45,25,sweet,dessert,حلو,تحلية,West Bengal,East


In [15]:
def create_qa_pairs(row):
  qa_pairs = []
  name = row.ARname
  ingredients = row.ARingredients
  diet = row.ARdiet
  prep_time = row.prep_time
  cook_time = row.cook_time
  state = row.state
  region = row.region
  flavor_profile = row.ARflavor_profile
  course = row.ARcourse

  qa_pairs.append({"question": f"ما هي المكونات اللازمة لتحضير {name}؟", "answer": ingredients})
  qa_pairs.append({"question": f"ما هي المكونات المطلوبة لعمل {name}؟", "answer": ingredients})
  qa_pairs.append({"question": f"مما يتكون {name}؟", "answer": ingredients})
  qa_pairs.append({"question": f"إيش هي مقادير {name}؟", "answer": ingredients})
  qa_pairs.append({"question": f"أعطني قائمة المحتويات لطبق {name}؟", "answer": ingredients})

# نوع الحمية (Diet)
  qa_pairs.append({"question": f"ما هو نوع الحمية الغذائية ل {name}؟", "answer": diet})
  qa_pairs.append({"question": f"هل {name} نباتي أم غير نباتي؟", "answer": diet})
  qa_pairs.append({"question": f"ما هو النظام الغذائي ل {name}؟", "answer": diet})
  qa_pairs.append({"question": f"هل يناسب {name} الأشخاص النباتيين؟", "answer": diet})
  qa_pairs.append({"question": f"تصنيف {name} من ناحية الدايت؟", "answer": diet})

# وقت التحضير (Prep Time)
  qa_pairs.append({"question": f"كم من الوقت يستغرق تحضير {name}؟", "answer": f"{prep_time} دقيقة"})
  qa_pairs.append({"question": f"ما هي مدة تحضير {name}؟", "answer": f"{prep_time} دقيقة"})
  qa_pairs.append({"question": f"كم دقيقة يستغرق تحضير {name}؟", "answer": f"{prep_time} دقيقة"})
  qa_pairs.append({"question": f"كم يبيلي وقت عشان أجهز {name}؟", "answer": f"{prep_time} دقيقة"})
  qa_pairs.append({"question": f"مدة تجهيز المكونات لـ {name} قبل الطبخ؟", "answer": f"{prep_time} دقيقة"})

# وقت الطهي (Cook Time)
  qa_pairs.append({"question": f"كم من الوقت يستغرق طهي {name}؟", "answer": f"{cook_time} دقيقة"})
  qa_pairs.append({"question": f"ما هي مدة طهي {name}؟", "answer": f"{cook_time} دقيقة"})
  qa_pairs.append({"question": f"كم دقيقة يستغرق طهي {name}؟", "answer": f"{cook_time} دقيقة"})
  qa_pairs.append({"question": f"كم يقعد {name} على النار؟", "answer": f"{cook_time} دقيقة"})
  qa_pairs.append({"question": f"ما هو الوقت اللازم لينضج {name} تماماً؟", "answer": f"{cook_time} دقيقة"})

# الولاية (State)
  qa_pairs.append({"question": f"ما هو أصل {name}؟", "answer": state})
  qa_pairs.append({"question": f"في أي ولاية تم اختراع {name}؟", "answer": state})
  qa_pairs.append({"question": f"إلى أي ولاية ينتمي {name}؟", "answer": state})
  qa_pairs.append({"question": f"أين نشأت أكلة {name} لأول مرة؟", "answer": state})
  qa_pairs.append({"question": f"ما هي الولاية المشهورة بتقديم {name}؟", "answer": state})

# المنطقة (Region)
  qa_pairs.append({"question": f"في أي منطقة يتم تحضير {name}؟", "answer": region})
  qa_pairs.append({"question": f"ما هي المنطقة التي ينتمي إليها {name}؟", "answer": region})
  qa_pairs.append({"question": f"في أي منطقة تشتهر {name}؟", "answer": region})
  qa_pairs.append({"question": f"من أي ناحية أو إقليم يأتي {name}؟", "answer": region})
  qa_pairs.append({"question": f"ما هو النطاق الجغرافي المعروف لطبق {name}؟", "answer": region})

# المذاق (Flavor Profile)
  qa_pairs.append({"question": f"ما هو المذاق الخاص ب {name}؟", "answer": flavor_profile})
  qa_pairs.append({"question": f"كيف يكون طعم {name}؟", "answer": flavor_profile})
  qa_pairs.append({"question": "ما هي نكهة {name}؟", "answer": flavor_profile})
  qa_pairs.append({"question": f"هل طعم {name} حلو أم حامض أم حار؟", "answer": flavor_profile})
  qa_pairs.append({"question": f"وصف لي نكهة {name} باختصار؟", "answer": flavor_profile})

# التصنيف الغذائي (Course)
  qa_pairs.append({"question": f"ما هو التصنيف الغذائي ل {name}؟", "answer": course})
  qa_pairs.append({"question": f"في أي وجبة يمكن تناول {name}؟", "answer": course})
  qa_pairs.append({"question": f"هل {name} يعد طبقاً رئيسياً أم جانبياً؟", "answer": course})
  qa_pairs.append({"question": f"بأي قسم في المنيو نلقى {name}؟", "answer": course})
  qa_pairs.append({"question": f"هل يُقدم {name} كمقبلات أم كوجبة أساسية؟", "answer": course})

  return qa_pairs

In [16]:
qa_list = df.apply(create_qa_pairs , axis=1).sum()
qa_df = pd.DataFrame(qa_list)
qa_df.head()

,question,answer
0,ما هي المكونات اللازمة لتحضير بالو شاهي؟,دقيق مائدة، زبادي، زيت، سكر
1,ما هي المكونات المطلوبة لعمل بالو شاهي؟,دقيق مائدة، زبادي، زيت، سكر
2,مما يتكون بالو شاهي؟,دقيق مائدة، زبادي، زيت، سكر
3,إيش هي مقادير بالو شاهي؟,دقيق مائدة، زبادي، زيت، سكر
4,أعطني قائمة المحتويات لطبق بالو شاهي؟,دقيق مائدة، زبادي، زيت، سكر


In [19]:
qa_df.to_csv(QA_file , index = False , encoding = 'UTF-8')

In [30]:
from sentence_transformers import SentenceTransformer
import numpy as np
model = SentenceTransformer('sentence-transformers/distiluse-base-multilingual-cased-v1')


def get_embeddings(text):
    embeddings = model.encode(text , convert_to_tensor=True)
    return embeddings.numpy()

q = qa_df.question.tolist()
q_emb = get_embeddings(q)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

In [31]:
import faiss

index = faiss.IndexFlatL2(q_emb.shape[1])
faiss.normalize_L2(q_emb)
index.add(q_emb)
faiss.write_index(index , QAfaiss_file)